In [ ]:
import os
import json
import torch
import warnings
from transformers import AutoTokenizer, AutoModelForCausalLM


MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

TRAIN_FILE = "msvd_train.json"
VAL_FILE = "msvd_val.json"
TEST_FILE = "msvd_test.json"

OUTPUT_DIR = "inference_outputs"

BATCH_SIZE = 4
MAX_NEW_TOKENS = 128
MAX_INPUT_LENGTH = 512

LIMIT_PER_SPLIT = 10

DO_SAMPLE = False
TEMPERATURE = 0.7
TOP_P = 0.9


warnings.filterwarnings("ignore")

assert torch.cuda.is_available(), "CUDA GPU not found."

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

DEVICE = "cuda"

print(f"Using GPU: {torch.cuda.get_device_name(0)}")

Using GPU: NVIDIA A100-SXM4-40GB


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
print(f"Loading tokenizer: {MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
    trust_remote_code=True,
)

# Important for decoder-only models
tokenizer.padding_side = "left"
tokenizer.truncation_side = "left"

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer padding_side:", tokenizer.padding_side)
print("Tokenizer pad_token:", tokenizer.pad_token)


print(f"Loading model: {MODEL_NAME}")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

model.config.pad_token_id = tokenizer.pad_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id

model.eval()

print("Model loaded.")

Loading tokenizer: mistralai/Mistral-7B-Instruct-v0.2
Tokenizer padding_side: left
Tokenizer pad_token: </s>
Loading model: mistralai/Mistral-7B-Instruct-v0.2


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model loaded.


In [ ]:
def find_caption_value(obj):
    possible_keys = [
        "caption",
        "captions",
        "sentence",
        "sentences",
        "text",
        "description",
        "descriptions",
        "query",
    ]

    if isinstance(obj, dict):
        for key in possible_keys:
            if key in obj:
                value = obj[key]

                if isinstance(value, str):
                    return value

                if isinstance(value, list):
                    if len(value) == 0:
                        return None

                    if isinstance(value[0], str):
                        return value[0]

                    if isinstance(value[0], dict):
                        return find_caption_value(value[0])

        for value in obj.values():
            found = find_caption_value(value)
            if found is not None:
                return found

    elif isinstance(obj, list):
        for item in obj:
            found = find_caption_value(item)
            if found is not None:
                return found

    return None


def flatten_items(data):
    if isinstance(data, list):
        return data

    if isinstance(data, dict):
        for key in ["annotations", "data", "items", "videos", "samples", "examples"]:
            if key in data and isinstance(data[key], list):
                return data[key]

        items = []

        for key, value in data.items():
            if isinstance(value, dict):
                item = dict(value)
                item["id"] = key
                items.append(item)

            elif isinstance(value, str):
                items.append({
                    "id": key,
                    "caption": value,
                })

            elif isinstance(value, list):
                for sub_value in value:
                    if isinstance(sub_value, str):
                        items.append({
                            "id": key,
                            "caption": sub_value,
                        })

                    elif isinstance(sub_value, dict):
                        item = dict(sub_value)
                        item["id"] = key
                        items.append(item)

        return items

    raise ValueError("Unsupported JSON format.")


def load_msvd_json(path):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Could not find {path}. Current folder files: {os.listdir('.')}"
        )

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    raw_items = flatten_items(data)

    rows = []

    for idx, item in enumerate(raw_items):
        caption = find_caption_value(item)

        if caption is None:
            continue

        video_id = None

        if isinstance(item, dict):
            for key in ["video_id", "video", "vid", "id", "name", "file_name", "filename"]:
                if key in item:
                    video_id = item[key]
                    break

        rows.append({
            "index": idx,
            "video_id": video_id,
            "caption": str(caption),
        })

    if len(rows) == 0:
        print("Could not automatically find captions.")
        print("Top-level JSON type:", type(data))

        if isinstance(data, dict):
            print("Top-level keys:", list(data.keys())[:20])
        elif isinstance(data, list) and len(data) > 0:
            print("First item:", data[0])

        raise ValueError("No captions found in JSON file.")

    return rows

In [ ]:
def build_prompt(caption):
    return (
        "You are given a video caption from the MSVD dataset.\n"
        "Rewrite it as a clean, concise normalized video description.\n\n"
        f"Caption: {caption}\n\n"
        "Normalized description:"
    )


def format_prompt(prompt):
    messages = [
        {
            "role": "user",
            "content": prompt,
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

In [ ]:
@torch.inference_mode()
def generate_batch(batch):
    prompts = [format_prompt(build_prompt(row["caption"])) for row in batch]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding="longest",
        truncation=True,
        max_length=MAX_INPUT_LENGTH,
        add_special_tokens=False,
    )

    input_ids = inputs["input_ids"].to(DEVICE)
    attention_mask = inputs["attention_mask"].to(DEVICE)

    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=DO_SAMPLE,
        temperature=TEMPERATURE if DO_SAMPLE else None,
        top_p=TOP_P if DO_SAMPLE else None,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )

    responses = []

    for i in range(outputs.shape[0]):
        generated_ids = outputs[i][input_ids.shape[1]:]
        response = tokenizer.decode(generated_ids, skip_special_tokens=True)
        responses.append(response.strip())

    return responses

In [ ]:
def run_split(split_name, file_path, limit=None, print_every=5):
    print("\n" + "=" * 60)
    print(f"Running inference for: {split_name}")
    print(f"Input file: {file_path}")
    print("=" * 60)

    rows = load_msvd_json(file_path)

    print(f"Total captions found: {len(rows)}")

    if limit is not None:
        rows = rows[:limit]
        print(f"Running inference on first {len(rows)} captions only.")
    else:
        print(f"Running inference on ALL {len(rows)} captions.")

    print(f"Example caption: {rows[0]['caption']}")

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    if limit is None:
        output_path = os.path.join(OUTPUT_DIR, f"{split_name}_outputs_all.jsonl")
    else:
        output_path = os.path.join(OUTPUT_DIR, f"{split_name}_outputs_{limit}.jsonl")

    with open(output_path, "w", encoding="utf-8") as out_f:
        for start in range(0, len(rows), BATCH_SIZE):
            batch = rows[start:start + BATCH_SIZE]
            responses = generate_batch(batch)

            for row, response in zip(batch, responses):
                result = {
                    "index": row["index"],
                    "video_id": row["video_id"],
                    "caption": row["caption"],
                    "model_output": response,
                }

                out_f.write(json.dumps(result, ensure_ascii=False) + "\n")

            done = min(start + BATCH_SIZE, len(rows))

            if done % print_every == 0 or done == len(rows):
                print(f"{split_name}: processed {done}/{len(rows)}")

    print(f"Saved output to: {output_path}")

    return output_path

In [ ]:
train_output = run_split("train", TRAIN_FILE, limit=None, print_every=5)
val_output = run_split("val", VAL_FILE, limit=None, print_every=5)
test_output = run_split("test", TEST_FILE, limit=None, print_every=5)

print("\nAll inference complete.")
print(train_output)
print(val_output)
print(test_output)


Running inference for: train
Input file: msvd_train.json
Total captions found: 1200
Running inference on ALL 1200 captions.
Example caption: a woman breaks an egg
train: processed 20/1200
train: processed 40/1200
train: processed 60/1200
train: processed 80/1200
train: processed 100/1200
train: processed 120/1200
train: processed 140/1200
train: processed 160/1200
train: processed 180/1200
train: processed 200/1200
train: processed 220/1200
train: processed 240/1200
train: processed 260/1200
train: processed 280/1200
train: processed 300/1200
train: processed 320/1200
train: processed 340/1200
train: processed 360/1200
train: processed 380/1200
train: processed 400/1200
train: processed 420/1200
train: processed 440/1200
train: processed 460/1200
train: processed 480/1200
train: processed 500/1200
train: processed 520/1200
train: processed 540/1200
train: processed 560/1200
train: processed 580/1200
train: processed 600/1200
train: processed 620/1200
train: processed 640/1200
train: p

In [ ]:
def preview_jsonl(path, n=10):
    print(f"\nPreviewing: {path}")
    print("=" * 60)

    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= n:
                break

            row = json.loads(line)

            print(f"\nItem {i + 1}")
            print("Video ID:", row["video_id"])
            print("Caption:", row["caption"])
            print("Output:", row["model_output"])


preview_jsonl(train_output, n=10)
preview_jsonl(val_output, n=10)
preview_jsonl(test_output, n=10)


Previewing: inference_outputs/train_outputs_all.jsonl

Item 1
Video ID: WTf5EgVY5uU_98_104
Caption: a woman breaks an egg
Output: A woman cracks an egg and separates its contents.

Item 2
Video ID: WeOU0Iba1Xg_1_30
Caption: a man galloping on his horse
Output: A man rides a galloping horse.

Item 3
Video ID: 0wutCy2ZGOQ_4_10
Caption: several boys are playing football in a fenced in area
Output: Several boys engage in a football game within a fenced area.

Item 4
Video ID: IWhrWLOAin0_1_4
Caption: a large dog gets something out of a refrigerator
Output: A large dog opens the refrigerator door and retrieves an item.

Item 5
Video ID: O9cOSO9L8Zs_1_16
Caption: a man is swinging on a rope
Output: A man performs a swinging motion on a rope.

Item 6
Video ID: 02Z-kuB3IaM_2_13
Caption: a black boar is running in the woods
Output: A black boar moves through the forest at a quick pace.

Item 7
Video ID: UbmZAe5u5FI_36_40
Caption: a woman is squeezing the liquid out of cucumber slices with her 

In [ ]:
import shutil
import os

folder_to_zip = "inference_outputs"
zip_name = "inference_outputs"

if not os.path.exists(folder_to_zip):
    raise FileNotFoundError(f"Folder not found: {folder_to_zip}")

zip_path = shutil.make_archive(zip_name, "zip", folder_to_zip)

print("Created zip file:", zip_path)

Created zip file: /content/inference_outputs.zip
